Бейзлайн для соревнования по поиску видеофрагментов.

Этапы:
1. Загружаем тестовые запросы и список аудиофайлов.
2. Транскрибируем все аудио с помощью Whisper turbo (сохраняем таймстемпы сегментов).
3. Разбиваем транскрипты на чанки (перекрывающиеся окна по 30 секунд).
4. Вычисляем эмбеддинги чанков с помощью e5-small-en-ru (с префиксом "passage: ").
5. Строим индекс FAISS.
6. Для каждого запроса вычисляем эмбеддинг (с префиксом "query: ") и ищем топ‑5 ближайших чанков.
7. Формируем файл сабмишна в формате sample_submission.csv.

In [1]:
# Устанавливаем необходимые библиотеки
!pip install openai-whisper==20250625
!pip install faiss-cpu==1.13.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=b775970fae7195fd22e50c8d2c29b45e776f347e80ba661874dbd978b8f93dfb
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.7 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
import torch
import whisper
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
import faiss
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_MODEL = "turbo"          # можно попробовать "tiny", "base", "small", "medium"
E5_MODEL = "d0rj/e5-small-en-ru"
HOME_DIR = Path("/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag/")     # папка с аудиофайлами (плоская структура)
VIDEO_FILES_CSV = HOME_DIR / "video_files.csv"   # список всех видео (не используется напрямую)
AUDIO_FILES_CSV = HOME_DIR / "audio_files.csv"   # список всех аудио
TEST_QUERIES = HOME_DIR / "test/test.csv"   # тестовые запросы
OUTPUT_SUBMISSION = "./submission.csv"           # наш результат
TRANSCRIPTS_PKL = HOME_DIR / "transcripts.pkl"

# Параметры чанкования
CHUNK_SECONDS = 30          # длина чанка в секундах
CHUNK_OVERLAP = 5           # перекрытие между чанками

## 1. Загрузка данных

In [4]:
print("Загружаем тестовые запросы...")
test_df = pd.read_csv(TEST_QUERIES)
print(f"Всего запросов: {len(test_df)}")

print("Загружаем список аудиофайлов...")
audio_df = pd.read_csv(AUDIO_FILES_CSV)
audio_paths = [HOME_DIR / p for p in audio_df["audio_path"].tolist()]
print(f"Всего аудиофайлов: {len(audio_paths)}")

Загружаем тестовые запросы...
Всего запросов: 812
Загружаем список аудиофайлов...
Всего аудиофайлов: 436


## 2. Транскрибация аудио с помощью Whisper

In [5]:
# print(f"Загружаем модель Whisper {WHISPER_MODEL}...")
# whisper_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)

# # Словарь для хранения результатов транскрибации: video_file -> список сегментов
# # Каждый сегмент: {"start": float, "end": float, "text": str}
# transcripts = {}

# def transcribe_audio(audio_path):
#     """Транскрибирует аудио и возвращает список сегментов с таймстемпами."""
#     # result = whisper_model.transcribe(str(audio_path), beam_size=5, word_timestamps=False, verbose=None)
#     result = whisper_model.transcribe(str(audio_path), word_timestamps=False, verbose=None)
#     segments = []
#     for seg in result["segments"]:
#         segments.append({
#             "start": seg["start"],
#             "end": seg["end"],
#             "text": seg["text"].strip()
#         })
#     return segments

# print("Транскрибируем аудиофайлы...")
# for audio_path in tqdm(audio_paths, desc="Транскрибация"):
#     if not audio_path.exists():
#         print(f"Предупреждение: {audio_path} не найден")
#         continue
#     # Из аудио пути получаем соответствующий видеофайл (заменяем audio/ на videos/ и префикс)
#     audio_name = audio_path.name
#     video_name = audio_name.replace("audio_", "video_")
#     video_file = f"videos/{video_name}"   # относительный путь для сабмишна
#     segments = transcribe_audio(audio_path)
#     transcripts[video_file] = segments

# print(f"Транскрибировано файлов: {len(transcripts)}")

In [6]:
with open(TRANSCRIPTS_PKL, "rb") as f:
    transcripts = pickle.load(f)

## 3. Разбиение на чанки

In [7]:
chunks = []   # список словарей: video_file, start, end, text
for video_file, segments in transcripts.items():
    if not segments:
        continue
    # Определим общую длительность
    total_end = segments[-1]["end"]
    current_start = 0.0
    while current_start < total_end - 1:
        chunk_end = min(current_start + CHUNK_SECONDS, total_end)
        # Собираем текст из сегментов, которые пересекаются с [current_start, chunk_end]
        chunk_text_parts = []
        for seg in segments:
            if seg["start"] < chunk_end and seg["end"] > current_start:
                chunk_text_parts.append(seg["text"])
        if chunk_text_parts:
            chunk_text = " ".join(chunk_text_parts)
            chunks.append({
                "video_file": video_file,
                "start": current_start,
                "end": chunk_end,
                "text": chunk_text
            })
        current_start += (CHUNK_SECONDS - CHUNK_OVERLAP)

print(f"Создано чанков: {len(chunks)}")

Создано чанков: 12880


## 4. Вычисление эмбеддингов для чанков с помощью E5

In [8]:
print(f"Загружаем модель E5 {E5_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL).to(DEVICE).eval()

def embed_texts(texts, prefix="passage: ", batch_size=32):
    """Вычисляет эмбеддинги для списка текстов с заданным префиксом."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = [prefix + t for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = e5_model(**inputs)
            # Используем эмбеддинг [CLS] (последнего слоя)
            embeddings = outputs.last_hidden_state[:, 0]  # (batch_size, hidden_size)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

# Тексты чанков
chunk_texts = [c["text"] for c in chunks]
print("Вычисляем эмбеддинги чанков...")
chunk_embeddings = embed_texts(chunk_texts, prefix="passage: ")

Загружаем модель E5 d0rj/e5-small-en-ru...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Вычисляем эмбеддинги чанков...


## 5. Построение индекса FAISS

In [9]:
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Индекс построен, размерность {dim}, число векторов {index.ntotal}")

Индекс построен, размерность 384, число векторов 12880


## 6. Обработка запросов и поиск

In [10]:
print("Вычисляем эмбеддинги запросов...")
queries = test_df["question"].tolist()
query_embeddings = embed_texts(queries, prefix="query: ", batch_size=32)

# Поиск топ‑5 для каждого запроса
k = 5
scores, indices = index.search(query_embeddings, k)

Вычисляем эмбеддинги запросов...


## 7. Формирование сабмишна

In [11]:
from pathlib import Path

# Создаём список строк: одна строка на каждый query_id
submission_rows = []
for qid in test_df["query_id"]:
    row = {"query_id": qid}
    # Инициализируем пустыми значениями для всех 5 рангов
    for rank in range(1, 6):
        row[f"video_file_{rank}"] = ""
        row[f"start_{rank}"] = 0.0
        row[f"end_{rank}"] = 0.0
    submission_rows.append(row)

# Словарь для быстрого доступа к индексу запроса
query_id_to_idx = {qid: i for i, qid in enumerate(test_df["query_id"])}

# Заполняем предсказания
for row in submission_rows:
    qid = row["query_id"]
    idx = query_id_to_idx.get(qid)
    if idx is not None:
        for rank in range(1, 6):
            if rank <= k and indices[idx, rank-1] != -1:
                chunk_idx = indices[idx, rank-1]
                chunk = chunks[chunk_idx]
                # Получаем имя видео без пути и расширения
                video_path = chunk["video_file"]
                video_name = Path(video_path).stem  # например, "video_abcd1234"
                row[f"video_file_{rank}"] = video_name
                row[f"start_{rank}"] = chunk["start"]
                row[f"end_{rank}"] = chunk["end"]
            # иначе остаются пустые, уже проставленные при инициализации

# Формируем итоговый DataFrame с правильным порядком колонок
cols = ["query_id"]
for rank in range(1, 6):
    cols.extend([f"video_file_{rank}", f"start_{rank}", f"end_{rank}"])
submission_df = pd.DataFrame(submission_rows)[cols]

submission_df.to_csv(OUTPUT_SUBMISSION, index=False, encoding="utf-8")
print(f"Сабмишн сохранён в {OUTPUT_SUBMISSION} (формат: одна строка на запрос, 5 предсказаний, видео без расширения)")

Сабмишн сохранён в ./submission.csv (формат: одна строка на запрос, 5 предсказаний, видео без расширения)
